# 06 · Независимая оценка CNN на RuASD
Поддерживаются CPU и GPU. Проверьте данные, затем включите `RUN_EVALUATION` и запустите ячейку вручную. Для `baseline_cached_v1` ожидается `best_model.pth` восьмой эпохи. Оценка использует параметры подготовки и порог, зафиксированные до просмотра тестовых результатов.

Метрики по исходным `sample_id` — основные. Результаты относятся к конкретным источникам и генераторам. Дикторская независимость не подтверждена. Если результат повлияет на настройки следующей модели, для её итоговой оценки потребуется новый независимый тест.

In [ ]:
# 1. Репозиторий и зависимости
import subprocess
import sys
from pathlib import Path

repo = Path('/content/audio-deepfake-detector')
url = 'https://github.com/Unvand/audio-deepfake-detector.git'
if not (repo / '.git').is_dir():
    subprocess.run(['git', 'clone', url, str(repo)], check=True)
else:
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only'], check=True)
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-r', str(repo / 'requirements-colab-train.txt')],
    check=True,
)
sys.path.insert(0, str(repo))
commit = subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True).strip()

In [ ]:
# 2. Drive и настройки оценки
import ipywidgets as widgets
import torch
from google.colab import drive, output
from IPython.display import display

from audio_deepfake_detector.src.cnn_training import resolve_run_directory
from audio_deepfake_detector.src.colab_cache import (
    cache_space_report,
    check_local_cache,
    prepare_local_cache,
)
from audio_deepfake_detector.src.ruasd_evaluation import (
    TEST_SPLIT,
    evaluate_unseen,
    evaluation_figures,
    evaluation_summary,
    read_test_manifest,
    save_evaluation,
)

drive.mount('/content/drive')
output.enable_custom_widget_manager()
base = Path('/content/drive/MyDrive/VKR_RuASD')
prepared = base / 'prepared_v1'
config_path = prepared / 'preprocessing_config.json'
cache = Path('/content/ruasd_test_unseen_cache')
device = 'cuda' if torch.cuda.is_available() else 'cpu'

run_id = widgets.Text(value='baseline_cached_v1', description='run_id')
batch = widgets.BoundedIntText(value=32, min=1, max=256, description='Batch size')
use_cache = widgets.Checkbox(value=False, description='Копировать test_unseen в /content')
save_results = widgets.Checkbox(value=True, description='Сохранить результаты на Drive')
status = widgets.Output()
check = widgets.Button(description='Проверить готовность')

def checkpoint_path():
    return resolve_run_directory(base / 'training_v1', run_id.value.strip()) / 'best_model.pth'

def checked_manifest():
    rows = read_test_manifest(prepared)
    records = rows.drop_duplicates('sample_id')
    expected = {
        ('fake', 'ruxTTS'): 500, ('fake', 'xtts_ru_ipa'): 500,
        ('real', 'CommonVoice'): 250, ('real', 'GOLOS'): 250,
        ('real', 'M-AILABS'): 250, ('real', 'SOVA-AB'): 250,
    }
    if len(rows) != 5149 or records.groupby(['label', 'subset']).size().to_dict() != expected:
        raise ValueError('Состав test_unseen не совпал с зафиксированным протоколом RuASD.')
    return rows

def show_status(_):
    with status:
        status.clear_output(wait=True)
        try:
            rows = checked_manifest()
            model_path = checkpoint_path()
            print('Устройство', device, '·', len(rows), 'сегментов')
            print('Checkpoint', model_path, '· найден' if model_path.is_file() else '· отсутствует')
            print('Конфигурация подготовки', 'найдена' if config_path.is_file() else 'отсутствует')
            print('test_unseen', (cache if use_cache.value else prepared) / TEST_SPLIT)
            if use_cache.value:
                report = cache_space_report(prepared, cache, (TEST_SPLIT,))
                print('Места для локального кэша достаточно' if report['enough_space']
                      else 'Места для локального кэша недостаточно')
            else:
                missing = sum(not (prepared / path).is_file() for path in rows['audio_path'])
                print('Отсутствующих FLAC', missing)
        except (OSError, ValueError) as error:
            print(error)

check.on_click(show_status)
display(widgets.VBox([run_id, batch, use_cache, save_results, check, status]))
print('Устройство определяется автоматически', device)
print('Checkpoint по умолчанию', checkpoint_path())
print('Тестовые FLAC на Drive', prepared / TEST_SPLIT)

In [ ]:
# 3. Синхронная оценка
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown

RUN_EVALUATION = False

if RUN_EVALUATION:
    checked_manifest()
    run = run_id.value.strip()
    checkpoint = checkpoint_path()
    expected_epoch = 8 if run == 'baseline_cached_v1' else None
    if not checkpoint.is_file() or not config_path.is_file():
        raise FileNotFoundError('Проверьте best_model.pth и preprocessing_config.json.')
    active_root = prepared
    if use_cache.value:
        cache_status = check_local_cache(
            prepared, cache, (TEST_SPLIT,), verify_checksums=True, compare_source_sizes=False,
        )
        if not cache_status.ready:
            copied = prepare_local_cache(prepared, cache, (TEST_SPLIT,), verify_checksums=True)
            if not copied.ready:
                raise RuntimeError(f'Кэш test_unseen не готов: {copied.errors[:5]}')
        active_root = cache
    result = evaluate_unseen(
        active_root, checkpoint, config_path, batch_size=batch.value,
        expected_epoch=expected_epoch, device=device, git_commit=commit, run_id=run,
    )
    figures = evaluation_figures(result)
    saved_to = None
    if save_results.value:
        saved_to = save_evaluation(result, figures, base / 'evaluation_v1' / run)
        print('Результаты сохранены в', saved_to)
    display(pd.DataFrame(result['metrics']).T)
    display(result['tables']['metrics_by_subset'])
    display(result['comparison'])
    display(Markdown(evaluation_summary(result)))
    for figure in figures.values():
        display(figure)
        plt.close(figure)

In [ ]:
# 4. Примеры ошибок из существующих FLAC
import soundfile as sf
from IPython.display import Audio

from audio_deepfake_detector.src.cnn_training import MelConfig, RuASDSegmentDataset

SHOW_ERROR_EXAMPLES = False

if SHOW_ERROR_EXAMPLES:
    if 'result' not in globals():
        raise RuntimeError('Сначала выполните оценку в ячейке 3.')
    cfg = result['config']
    root = Path(cfg['prepared_root'])
    mel_config = MelConfig(**cfg['mel_config'])
    rows = read_test_manifest(root)
    display(Markdown(
        'Показан первый четырёхсекундный сегмент каждой выбранной записи. '
        'Оценка в таблице усреднена по всем её сегментам. '
        'Спектрограмма сама по себе не объясняет причину ошибки.'
    ))
    for key, title in (
        ('false_positives', 'Ложные тревоги'), ('false_negatives', 'Пропуски'),
    ):
        examples = result['tables'][key].head(3)
        display(Markdown(f'### {title}'))
        display(examples)
        for number, example in enumerate(examples.itertuples(index=False), start=1):
            waveform, rate = sf.read(root / example.audio_path)
            display(Audio(waveform.T, rate=rate, normalize=False))
            selected = rows.loc[rows['audio_path'].eq(example.audio_path)]
            dataset = RuASDSegmentDataset(selected, root, TEST_SPLIT, mel_config)
            mel_db, _ = dataset.visualization_item(0)
            figure, ax = plt.subplots(figsize=(8, 3), layout='constrained')
            duration = mel_db.shape[-1] * mel_config.hop_length / mel_config.sample_rate
            image = ax.imshow(mel_db.numpy(), origin='lower', aspect='auto',
                              extent=(0, duration, 0, mel_config.n_mels), cmap='magma')
            ax.set(title=f'Log-Mel · {example.subset} · {example.true_class} → {example.predicted_class}',
                   xlabel='Время, с', ylabel='Mel-полоса')
            figure.colorbar(image, ax=ax, label='дБ')
            if saved_to is not None:
                figure_path = saved_to / f'{key}_mel_{number:02}.png'
                if not figure_path.exists():
                    figure.savefig(figure_path, dpi=160)
            plt.show()
            plt.close(figure)

## Добавить результат в журнал ВКР
Скачайте `evaluation_summary.md` из сохранённых результатов и проверьте его. Из корня репозитория выполните `python scripts/build_vkr_report.py --append-evaluation /путь/evaluation_summary.md`. Журнал сохранит строки обучения и добавит оценку отдельной записью.